# 7. Downstream of a Failure (Blast Radius)
**Difficulty:** 🟡 Medium · **Topic:** Graphs / BFS · DFS · **Pattern:** Reachability from a source (visited-set traversal)

> **DevRev context:** services and tasks form a **dependency graph** — an arrow `u → v` means *v depends on u* (v runs after u, or v consumes u's output). When `u` **fails**, everything that depends on it, directly or transitively, is affected. That set is the **blast radius**, and a single **BFS/DFS from the failed node** finds it.

## 💡 Concepts

**Core concept(s):** **Graph reachability** — from a starting node, visit everything you can reach by following arrows, using a **visited set** so you never revisit (and never loop forever on a cycle).

**Why it applies here:** "Which tasks are affected by this failure?" is exactly "which nodes are **reachable** from the failed node, following the dependency arrows?" Both **BFS** (a queue) and **DFS** (a stack/recursion) answer it in one sweep.

**Key intuition:** Start at the failure, mark it, and fan out to its dependents; each newly-reached task's dependents are affected too. Stop when nothing new is reachable.

---

### 📚 What is a directed graph / adjacency list?
A **directed graph** is nodes joined by one-way arrows. An **adjacency list** stores, for each node, the list of nodes its arrows point to — here, `u -> [its dependents]`. Building it once turns "who depends on u?" into an O(1) lookup.

### 📚 What are BFS and DFS?
Both explore a graph from a start node.
- **BFS (Breadth-First Search)** uses a **queue** — it visits in rings: the failure, then its direct dependents, then *their* dependents. The ring number = **how many hops** from the failure (a natural *impact order*).
- **DFS (Depth-First Search)** uses a **stack / recursion** — it dives down one dependency chain fully before backing up.
For "what set is reachable?" they give the **same answer**; pick BFS when you also want distance/impact-order.

### 📚 Why a visited set?
Real dependency graphs can have cycles (A waits on B waits on A, misconfigured). A **visited set** guarantees each node is processed once → the sweep is **O(V + E)** and can never loop forever.

---

**Prerequisite knowledge:**
- Adjacency-list construction from an edge list.
- A queue (BFS) or stack (DFS) frontier.
- Edge **direction**: downstream (dependents) vs upstream (dependencies) — reverse the arrows to trace a root cause.

## 📝 Problem

Given `n` tasks (`0..n-1`) and directed dependency edges `[u, v]` meaning **`u` unlocks `v`** (v depends on u),
and a **failed** node `source`, return the set of **all downstream tasks affected** — everything reachable
from `source` — **excluding** `source` itself.

**Example**
```
n = 6
edges = [[0,1],[0,2],[1,3],[1,4],[2,4],[4,5]]
        Auth→API, Auth→Billing, API→Notif, API→Dashboard, Billing→Dashboard, Dashboard→Reports

source = 1 (API fails)  ->  {3, 4, 5}   (Notif, Dashboard, Reports)
        Auth(0) and Billing(2) are UPSTREAM/sibling -> unaffected

source = 0 (Auth fails) ->  {1, 2, 3, 4, 5}   (everything)
source = 5 (Reports)    ->  set()             (nothing depends on it)
```

> Two approaches: a naive **re-scan-every-edge fixed point** `O(V·E)` and a single **BFS/DFS** `O(V+E)`.

### Approach 1 — Re-scan Every Edge Until Stable (worst)

Keep a set of "reached" nodes seeded with the failure. Repeatedly scan **all** edges: if an edge starts at a
reached node, its endpoint becomes reached. Repeat until a full pass adds nothing. Correct, but it re-walks
every edge each round → `O(V·E)`.

In [ ]:
from typing import List, Set

def downstream_naive(n: int, edges: List[List[int]], source: int) -> Set[int]:
    reach = {source}                       # nodes known to be affected (incl. the failure)
    changed = True
    while changed:                         # keep going until a full pass changes nothing
        changed = False
        for u, v in edges:                 # re-scan EVERY edge every round (the slow part)
            if u in reach and v not in reach:
                reach.add(v)               # v depends on an affected node -> affected
                changed = True
    reach.discard(source)                  # report downstream only (exclude the failed node)
    return reach

### Approach 2 — BFS / DFS From the Failure (optimal)

Build the adjacency list once, then sweep outward from `source` with a **visited set**. BFS uses a queue
(and yields hop-distance = impact order); DFS uses a stack. Each node and edge is touched once → `O(V+E)`.

In [ ]:
from typing import List, Set
from collections import deque, defaultdict

def _build_adj(n: int, edges: List[List[int]]):
    adj = defaultdict(list)                # u -> list of nodes that depend on u
    for u, v in edges:
        adj[u].append(v)
    return adj

def downstream_bfs(n: int, edges: List[List[int]], source: int) -> Set[int]:
    adj = _build_adj(n, edges)
    seen = {source}                        # visited set: cycle-safe, process each node once
    q = deque([source])                    # frontier of nodes whose dependents we still owe
    while q:
        u = q.popleft()                    # BFS: FIFO -> explores in rings (hop order)
        for v in adj[u]:                   # every task that depends on u
            if v not in seen:
                seen.add(v)                # newly affected
                q.append(v)                # its dependents are next
    seen.discard(source)                   # downstream only
    return seen

def downstream_dfs(n: int, edges: List[List[int]], source: int) -> Set[int]:
    adj = _build_adj(n, edges)
    seen = {source}
    stack = [source]                       # DFS: same reachability, LIFO frontier
    while stack:
        u = stack.pop()                    # dive down one chain before backing up
        for v in adj[u]:
            if v not in seen:
                seen.add(v)
                stack.append(v)
    seen.discard(source)
    return seen

In [ ]:
# Correctness check
n = 6
edges = [[0, 1], [0, 2], [1, 3], [1, 4], [2, 4], [4, 5]]

# API (1) fails -> Notif(3), Dashboard(4), Reports(5) affected; Auth/Billing are not.
assert downstream_naive(n, edges, 1) == {3, 4, 5}
assert downstream_bfs(n, edges, 1) == {3, 4, 5}
assert downstream_dfs(n, edges, 1) == {3, 4, 5}

# Auth (0) fails -> everything downstream.
assert downstream_bfs(n, edges, 0) == {1, 2, 3, 4, 5}
# A leaf fails -> nothing depends on it.
assert downstream_bfs(n, edges, 5) == set()

# Cycle-safe: 0->1->2->0, fail 0 -> {1, 2} (visited set stops the loop).
cyc = [[0, 1], [1, 2], [2, 0]]
assert downstream_naive(3, cyc, 0) == {1, 2}
assert downstream_bfs(3, cyc, 0) == {1, 2}
assert downstream_dfs(3, cyc, 0) == {1, 2}

print("All tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio `n`→`2n` |
|---|---|
| `O(n)` / `O(V+E)` | ≈ **2×** |
| `O(n log n)`      | ≈ **2×** (slightly more) |
| `O(V·E)` ~ `O(n²)`| ≈ **4×** |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    # A single long chain 0->1->2->...->(n-1), failure at the head.
    # Edges are listed in REVERSE order ([n-2,n-1] ... [0,1]) on purpose: that way a
    # single edge-scan can only propagate ONE hop, so the naive fixed point needs
    # ~n full passes x E edges = O(V*E) ~ O(n^2). (Ascending edges would let one pass
    # cascade the whole chain and hide the quadratic cost.)
    # BFS/DFS reaches all n nodes in one linear sweep regardless -> O(n).
    edges = [[i, i + 1] for i in range(n - 2, -1, -1)]
    return (n, edges, 0)

solutions = {
    "re-scan  O(V*E)": downstream_naive,
    "BFS      O(V+E)": downstream_bfs,
}
sizes = [200, 400, 800, 1600]

benchmark(solutions, make_worst_case, sizes, plot=True)

## 🧩 Patterns Learned

- **Blast radius = reachability from a source:** one BFS or DFS with a **visited set** finds every affected node in `O(V+E)`.
- **BFS gives impact order:** the ring number = hops from the failure, so you can report "directly hit" vs "two steps away".
- **Direction is the whole game:** follow arrows for **downstream** (who's affected); **reverse** the arrows for **upstream** (root-cause / what this task waited on).
- **Visited set = cycle safety:** without it, a dependency loop spins forever; with it, each node is processed once.
- **Signal:** "who is affected / blast radius / cascading failure / everything downstream of X".
- **DevRev / related:** incident impact analysis, dependency/blast-radius on a service or workflow graph; same visited-set traversal as Number of Islands and Clone Graph.
- **Common pitfalls:** (1) dropping the visited set (infinite loop on cycles); (2) wrong edge direction (downstream vs upstream); (3) inconsistently including/excluding the source node.